# 13. Trip Inspection — Linie 8 / 11 / 61 (2025-07-31)

Browse real detected stop events for three DVB lines on one service day.  
Times displayed in **Europe/Berlin** (CEST = UTC+2).

In [1]:
from pathlib import Path
from datetime import date
import polars as pl

DATA = Path('../data')

# ── Stop name lookup (one canonical name per ort_nr) ─────────────────────────
stop_names = (
    pl.read_csv(DATA / 'stop_geometry.csv')
    .select(['ort_nr', 'stop_name'])
    .group_by('ort_nr')
    .agg(pl.col('stop_name').first())
)

# ── Load events, add Berlin local-time columns ────────────────────────────────
df = (
    pl.read_parquet(DATA / 'processed_with_signal_info/core_stop_events.parquet')
    .with_columns(
        pl.col('arrival_time').dt.convert_time_zone('Europe/Berlin').alias('arrival_local'),
        pl.col('scheduled_arrival_time').dt.convert_time_zone('Europe/Berlin').alias('sched_local'),
    )
)

print(f'Loaded {len(df):,} stop events')
print(f'Date range (local): {df["arrival_local"].min().date()} → {df["arrival_local"].max().date()}')

Loaded 950,531 stop events
Date range (local): 2025-07-28 → 2025-08-03


In [4]:
import json

# ── 加载原始时刻表，每个 fahrt_id 保留最新快照 ────────────────────────────────
raw_tt = (
    pl.read_csv(DATA / 'timetable_trips_2025_07_22.csv', infer_schema_length=1000)
    .with_columns(pl.col('tst_iso').str.to_datetime(format='%Y-%m-%dT%H:%M:%S%.f%z'))
    .sort('tst_iso', descending=True)
    .unique(subset=['fahrt_id'], keep='first')
)
print(f'时刻表快照: {len(raw_tt):,} 个 fahrt_id')


def print_planned_route(fahrt_id):
    """打印时刻表中该 fahrt_id 的完整计划站点序列（相对出发时间）。"""
    rows = raw_tt.filter(pl.col('fahrt_id').cast(pl.Int64) == fahrt_id)
    if len(rows) == 0:
        print(f'  (fahrt_id={fahrt_id} 不在时刻表快照中)')
        return
    try:
        segments = json.loads(rows['segmente'][0])
    except Exception:
        print('  (segmente 解析失败)')
        return
    if not segments:
        print('  (segmente 为空)')
        return

    print(f'\n  ┌── 计划路线  {len(segments)} 站（时间相对出发时刻）')
    print(f'  │   {"#":>3}  {"Haltestelle":<34}  {"ort_nr":>8}  {"+时间":>7}')
    print(f'  │   {"─"*56}')
    cumulative = 0
    for i, seg in enumerate(segments):
        ort  = seg.get('ort_nr')
        nr   = stop_names.filter(pl.col('ort_nr') == ort)
        name = (nr['stop_name'][0] if len(nr) > 0 else None) or f'ort_nr={ort}'
        name = name[:34]
        mins, secs = divmod(int(cumulative), 60)
        branch = '  └──' if i == len(segments) - 1 else '  │  '
        print(f'{branch}  {i:>3}  {name:<34}  {ort!s:>8}  +{mins}:{secs:02d}')
        cumulative += seg.get('lenkzeit', 0)

时刻表快照: 97,982 个 fahrt_id


In [5]:
TARGET_DAY   = date(2025, 7, 31)   # Berlin local date
TARGET_LINES = [8, 11, 61]

# ── Filter to target day & lines ──────────────────────────────────────────────
day = df.filter(
    (pl.col('arrival_local').dt.date() == TARGET_DAY) &
    pl.col('linie').is_in(TARGET_LINES)
)

# ── Trip-level summary ────────────────────────────────────────────────────────
trip_summary = (
    day
    .group_by(['linie', 'fahrt_id'])
    .agg(
        pl.len().alias('n_stops'),
        pl.col('fzg_id').first().alias('fzg_id'),
        pl.col('arrival_local').min().alias('start_time'),
        pl.col('arrival_local').max().alias('end_time'),
        pl.col('delay_calculated_sec').mean().round(1).alias('avg_delay_s'),
    )
    .sort(['linie', 'start_time'])
)

print(f'Day: {TARGET_DAY}  (Berlin local time)\n')
print(f'{"Line":>6}  {"Trips":>6}  {"Stops range":>12}  {"Operating hours"}')
print('─' * 55)
for linie in TARGET_LINES:
    t = trip_summary.filter(pl.col('linie') == linie)
    if len(t) == 0:
        print(f'{linie:>6}  {"—":>6}  (no data)')
        continue
    print(f'{linie:>6}  {len(t):>6}  '
          f'{t["n_stops"].min():>4}–{t["n_stops"].max():<4} stops  '
          f'{t["start_time"].min().strftime("%H:%M")} – {t["end_time"].max().strftime("%H:%M")}')

Day: 2025-07-31  (Berlin local time)

  Line   Trips   Stops range  Operating hours
───────────────────────────────────────────────────────
     8     154    13–37   stops  00:00 – 23:59
    11     235     3–31   stops  00:00 – 23:59
    61     327     6–42   stops  00:00 – 23:58


In [6]:
def print_trip(linie, fahrt_id):
    trip = (
        day
        .filter((pl.col('linie') == linie) & (pl.col('fahrt_id') == fahrt_id))
        .sort('stop_index')
        .join(stop_names, left_on='ort_nr_start', right_on='ort_nr', how='left')
    )
    if len(trip) == 0:
        return

    fzg   = trip['fzg_id'][0]
    t0    = trip['arrival_local'][0].strftime('%H:%M')
    t1    = trip['arrival_local'][-1].strftime('%H:%M')
    n_ok  = trip['stop_name'].is_not_null().sum()
    print(f'\n  Linie {linie}  │  fahrt_id={fahrt_id}  │  Fahrzeug {fzg}  │  {t0} → {t1}  ({len(trip)} Haltestellen, {n_ok} mit Namen)')
    print(f'  {"#":>3}  {"Haltestelle":<34}  {"Plan":>8}  {"Ist":>8}  {"Delay":>7}  Status')
    print(f'  {"─"*72}')

    for r in trip.iter_rows(named=True):
        name   = (r['stop_name'] or f'ort_nr={r["ort_nr_start"]}')[:34]
        sched  = r['sched_local'].strftime('%H:%M:%S')  if r['sched_local']  is not None else '   —    '
        actual = r['arrival_local'].strftime('%H:%M:%S')
        d      = r['delay_calculated_sec']
        delay  = f'{d:+.0f}s' if d is not None else '  n/a'
        status = r['stop_status'] or ''
        print(f'  {r["stop_index"]:>3}  {name:<34}  {sched:>8}  {actual:>8}  {delay:>7}  {status}')


N_PER_LINE = 2   # trips shown per line (1 morning + 1 afternoon when possible)

for linie in TARGET_LINES:
    trips = trip_summary.filter(pl.col('linie') == linie)
    if len(trips) == 0:
        continue

    morning   = (trips.filter(pl.col('start_time').dt.hour() < 12)
                      .sort('n_stops', descending=True).head(1))
    afternoon = (trips.filter(pl.col('start_time').dt.hour() >= 12)
                      .sort('n_stops', descending=True).head(1))

    print(f'\n{"═"*76}')
    print(f'  LINIE {linie}  ·  {len(trips)} Fahrten am {TARGET_DAY}')
    print(f'{"═"*76}')

    shown = set()
    for chunk in [morning, afternoon]:
        if len(chunk) == 0:
            continue
        row = chunk.row(0, named=True)
        fid = row['fahrt_id']
        if fid not in shown:
            shown.add(fid)
            print_trip(linie, fid)
            print_planned_route(fid)


════════════════════════════════════════════════════════════════════════════
  LINIE 8  ·  154 Fahrten am 2025-07-31
════════════════════════════════════════════════════════════════════════════

  Linie 8  │  fahrt_id=16884130  │  Fahrzeug 854  │  00:11 → 01:00  (37 Haltestellen, 37 mit Namen)
    #  Haltestelle                             Plan       Ist    Delay  Status
  ────────────────────────────────────────────────────────────────────────
    0  Brunnenweg                          00:10:00  00:11:25     +86s  no_door
    1  Festspielhaus Hellerau              00:11:00  00:12:34     +95s  no_door
    2  Heinrich-Tessenow-Weg               00:12:00  00:13:23     +84s  no_door
    3  Am Hellerrand                       00:13:00  00:14:07     +68s  no_door
    4  Infineon Süd                        00:14:00  00:16:32    +153s  no_door
    5  Moritzburger Weg                    00:16:00  00:17:20     +81s  no_door
    6  Hellersiedlung                      00:17:00  00:18:31     +92s

## Explore any trip

Change `LINIE` / `FAHRT_ID` to inspect any specific trip, or use `trip_summary` to browse.

In [ ]:
# Browse all trips for a line — sorted by start time
BROWSE_LINIE = 8

print(f'All Linie {BROWSE_LINIE} trips on {TARGET_DAY}:\n')
print(f'  {"fahrt_id":>12}  {"fzg":>5}  {"Start":>6}  {"End":>6}  {"Stops":>6}  {"Avg delay":>10}')
print(f'  {"─"*55}')
for row in trip_summary.filter(pl.col('linie') == BROWSE_LINIE).iter_rows(named=True):
    print(f'  {row["fahrt_id"]:>12}  {row["fzg_id"]:>5}  '
          f'{row["start_time"].strftime("%H:%M"):>6}  {row["end_time"].strftime("%H:%M"):>6}  '
          f'{row["n_stops"]:>6}  {row["avg_delay_s"]:>9.1f}s')

In [ ]:
# ── 查看某一条具体 trip ──────────────────────────────────────────────────────
LINIE    = 8
FAHRT_ID = trip_summary.filter(
    (pl.col('linie') == LINIE) & (pl.col('n_stops') == trip_summary.filter(pl.col('linie') == LINIE)['n_stops'].max())
)['fahrt_id'][0]   # 默认：line 8 中检测到站点最多的 trip

print_trip(LINIE, FAHRT_ID)
print_planned_route(FAHRT_ID)